# TD10d - Quantization

In [1]:
import torch

# define a random model that takes 4000 input features and outputs 4 features with an intermediate layer of 4000 features
class M(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = torch.nn.Linear(4000, 4000)
        self.fc2 = torch.nn.Linear(4000, 4)

    def forward(self, x):
        x = self.fc(x)
        x = self.fc2(x)
        return x

# create a model instance
model_fp32 = M()

# create a quantized model instance
model_int8 = torch.ao.quantization.quantize_dynamic(
    model_fp32,  # the original model
    {torch.nn.Linear},  # a set of layers to dynamically quantize
    dtype=torch.qint8  # the target dtype for quantized weights, here we use 8-bit integer - we are going a bit crazy on the quantization amount
)

In [2]:
# run the model with and without quantization on a batch of four random inputs - let's see the time difference
input_ = torch.randn(4, 4000)
with torch.no_grad():
    %timeit res_32 = model_fp32(input_)
    %timeit res_16 = model_int8(input_)

7.42 ms ± 182 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
438 µs ± 14 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [3]:
# Make sure the results are roughly the same because if it's faster but outputs stupid stuff, it's not really worth it
with torch.no_grad():
    res_32 = model_fp32(input_)
    res_16 = model_int8(input_)
print(res_16, "\n", res_32)

tensor([[ 0.0425,  0.4171, -0.0090,  0.1418],
        [-0.0166,  0.2736,  0.4394,  0.2597],
        [ 0.1512,  0.1344, -0.1763,  0.2258],
        [-0.4419, -1.0827, -0.6185, -0.4859]]) 
 tensor([[ 0.0430,  0.4132, -0.0049,  0.1381],
        [-0.0274,  0.2756,  0.4294,  0.2490],
        [ 0.1490,  0.1215, -0.1847,  0.2257],
        [-0.4432, -1.0862, -0.6118, -0.4720]])


In [4]:
# save both models
torch.save(model_fp32.state_dict(), "model_fp32.pth")
torch.save(model_int8.state_dict(), "model_int8.pth")

And now, compare the sizes of the two models on your hard drive.